<a href="https://colab.research.google.com/github/890tsugua/AugustBHilger/blob/main/Diffusion_Project_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!rm md_header.py
!wget https://www.dropbox.com/s/vnyydp0qcl1yid5/md_header.py

# python modules
import numpy as np
import copy
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.stats import linregress
# local modules
import md_header as md

rm: cannot remove 'md_header.py': No such file or directory
--2024-06-11 14:13:54--  https://www.dropbox.com/s/vnyydp0qcl1yid5/md_header.py
Resolving www.dropbox.com (www.dropbox.com)... 162.125.5.18, 2620:100:601d:18::a27d:512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.5.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /scl/fi/2kp3s1y9rvi98cve9c75f/md_header.py?rlkey=0xhjtgzk6dlgqskdn759hvkgg [following]
--2024-06-11 14:13:55--  https://www.dropbox.com/scl/fi/2kp3s1y9rvi98cve9c75f/md_header.py?rlkey=0xhjtgzk6dlgqskdn759hvkgg
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc291518aca4698fb34ef2403853.dl.dropboxusercontent.com/cd/0/inline/CUoe5RM5KernzQ1GqffWWxSbhMfOHIQreGriCJFsCEx8tyBhwsQ8m6QB0MJgVJJ-VHPlWO00nIyjXo2fkKeqRl4f0jl8mocpLp8xxVY8A6lQVbshnFbxIZSouTO0hlVQvv8/file# [following]
--2024-06-11 14:13:55--  https://uc291518aca4698fb34ef2403853.dl.dropboxusercontent.c

/content/md_header.py:112: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit()


# Defining functions

In [ ]:
# Calculate msd from a value of R compared to R0
def get_msd(R, R0, n_particles):
  displacement = R - R0
  # Computing the mean squared displacement
  msd = np.vdot(displacement, displacement)/n_particles
  return msd

In [ ]:
# Calculate vacf from a value of V compated to V0
def get_vacf(V, V0, n_particles):
  vacf = np.sum(V * V0)/n_particles
  return vacf

In [ ]:
# Calculate both at the same time
def get_D(msd_list, vacf_list):
  time = np.linspace(0,1,t_max)
  D_msd = stats.linregress(time, msd_list)[0]/4
  D_vacf = np.trapz(vacf_list, x = time)/2
  return D_msd, D_vacf

In [ ]:
# Calculate diffusion coefficient from msd
def get_D_msd(time, msd_list):
  result = linregress(time, msd_list)
  # The slope of the line ([0]) is divided by 4
  # to get the diffusion coefficient
  D = result[0]/4
  return D

In [ ]:
# Calculate diffusion coefficient from vacf
def get_D_msd(time, vacf_list):
  # We integrate over y with respect to x
  D = 0.5 * np.trapz(vacf_list, x = time)
  return D

In [ ]:
# Calculate correlated msd and vacf
def get_msd_vacf_cor(R_list, V_list):

  msd_list = [0 for i in range(t_max)]
  vacf_list = [0 for i in range(t_max)]

  for t in range(t_max):
    for t0 in range(n_samples - t):
      msd_list[t] += get_msd(R_list[t0+t], R_list[t0], n_particles)
      vacf_list[t] += get_vacf(V_list[t0+t], V_list[t0], n_particles)

    msd_list[t] /= (n_samples - t)
    vacf_list[t] /= (n_samples - t)

  return msd_list, vacf_list

In [ ]:
# Calculate uncorrelated msd and vacf
def get_msd_vacf_uncor(R_list, V_list):
  msd_un = []
  vacf_un = []

  for n in range(len(R_list)):
    # Calculating msd an vacf for the funtion
    msd = get_msd(R_list[n], R_list[0], n_particles)
    msd_un.append(msd)
    vacf = get_vacf(V_list[n], V_list[0], n_particles)
    vacf_un.append(vacf)

  return msd_un, vacf_un

In [ ]:
# Velo-verlet function
def velo_verlet(R, V, F, box_width, n_particles, dt, eps):

    F_old = copy.copy(F)

    # Step 1: Update positions
    #V[abs(R) > box_width] *= -1
    R += V*dt + 0.5*F*dt*dt

    # Step 2: Update forces
    energy, F = md.lennard_jones(R, box_width, eps)

    # Step 3: Update velocities
    V += 0.5*(F + F_old)*dt

    return R, V, F, energy

In [ ]:
# Calculate kinetic energy
def kinetic(V):
  ke = 0.5*np.sum(V**2)

  return ke

In [ ]:
# Scale temperature
def scale_temp(V,T,n_particles):
  ke = kinetic(V)
  T_current = ke/n_particles
  c = np.sqrt(T/T_current)
  V *= c

  return V

In [ ]:
# Simulation function
def simulation(n_particles, n_steps, dt, rho, T, sample_freq):

  ## Initializing variables
  R_list = []
  V_list = []
  t_list = []
  n_samples = 0
  eps = np.ones((n_particles, n_particles))

  ## Initialize particles
  R, V, F, box_width = md.initialize_particles(n_particles, T, rho, eps)


  ## Take simulation steps
  for n in range(n_steps):

    # Updating positions, velocities and forces for particles
    R, V, F, potential_energy = velo_verlet(R, V, F, box_width, n_particles,
                                            dt, eps)

    # Scaling velicities to match temperature
    if n % 5 == 0:
      V = scale_temp(V, T, n_particles)

    # Saving positions
    if n >= n_eq:
      if n % sample_freq == 0:
        R_list.append(copy.copy(R))
        V_list.append(copy.copy(V))
        t_list.append(copy.copy(n * dt))
        n_samples += 1

  return R_list, V_list, t_list, n_samples

In [ ]:
""" IKKE BRUGT
# Save multiple diffusion coefficients
def run(n_times):
  D_msd = []
  D_vacf = []

  for i in range(n_times):
    R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, rho, T, sample_freq)
    msd_list_temp, vacf_list_temp = get_msd_vacf_cor(R_list, V_list)
    D_msd_temp, D_vacf_temp = get_D(msd_list_temp, vacf_list_temp)
    D_msd.append(D_msd_temp)
    D_vacf.append(D_vacf_temp)
  return D_msd, D_vacf


SyntaxError: incomplete input (<ipython-input-14-1b972ab5e560>, line 1)

In [ ]:
""" IKKE BRUGT
# Save multiple diffusion coefficients
def run(reps):
  D_msd = []
  D_vacf = []

  for i in range(reps):
    R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, rho, T, sample_freq)
    msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)
    D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)
    D_msd.append(D_msd_temp)
    D_vacf.append(D_vacf_temp)
  return D_msd, D_vacf


# Running simulations

In [ ]:
## Simulation Initialization
n_particles = 108
T = 0.5
rho = 0.7
sample_freq = 24
n_eq = 10_000
n_steps = 20_000
dt = 0.001
eps = np.ones((n_particles, n_particles))
t_max = 45

In [ ]:
## Simulation Initialization
n_particles = 108
T = 0.5
rho = 0.7
sample_freq = 24
n_eq = 10_000
n_steps = 20_000
dt = 0.001
eps = np.ones((n_particles, n_particles))
t_max = 45

# @title Simulation 1: different n_steps
n_steps = list(range(20_000, 150_000, 5000))
n_particles = 108
reps = 10

data = []

for n in n_steps:
  print(f'Calculating n_steps = {n}')
  D_msd_list = []
  D_vacf_list = []

  for i in range(reps):
    R_list, V_list, t_list, n_samples = simulation(n_particles, n, dt, rho, T, sample_freq)
    msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)
    D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)
    D_msd_list.append(D_msd_temp)
    D_vacf_list.append(D_vacf_temp)

  data.append({'n_steps': n, 'D_msd': D_msd_list, 'D_vacf': D_vacf_list})

# Convert to DataFrame
df = pd.DataFrame(data)

## Display the DataFrame or save to CSV file for excel
df.to_csv('simulation_1.csv', index=False)

Calculating n_steps = 20000
Calculating n_steps = 25000
Calculating n_steps = 30000
Calculating n_steps = 35000
Calculating n_steps = 40000
Calculating n_steps = 45000
Calculating n_steps = 50000
Calculating n_steps = 55000
Calculating n_steps = 60000
Calculating n_steps = 65000
Calculating n_steps = 70000
Calculating n_steps = 75000
Calculating n_steps = 80000
Calculating n_steps = 85000
Calculating n_steps = 90000
Calculating n_steps = 95000
Calculating n_steps = 100000
Calculating n_steps = 105000
Calculating n_steps = 110000
Calculating n_steps = 115000
Calculating n_steps = 120000
Calculating n_steps = 125000
Calculating n_steps = 130000
Calculating n_steps = 135000
Calculating n_steps = 140000
Calculating n_steps = 145000


In [ ]:
## Simulation Initialization
n_particles = 108
T = 0.5
rho = 0.7
sample_freq = 24
n_eq = 10_000
n_steps = 20_000
dt = 0.001
eps = np.ones((n_particles, n_particles))
t_max = 45

# @title Simulation 2: different n_particles
n_particles_list = list(range(50, 201, 10))
n_steps = 20_000
reps = 10

data = []

for n in tqdm(n_particles_list):
  #print(f'Calculating n_particles = {n}')
  n_particles = n
  D_msd = []
  D_vacf = []
  for i in range(reps):
    R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, rho, T, sample_freq)
    msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)
    D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)
    D_msd.append(D_msd_temp)
    D_vacf.append(D_vacf_temp)
  data.append({'n_particles': n, 'D_msd': D_msd, 'D_vacf': D_vacf})

df = pd.DataFrame(data)

## Display the DataFrame or save to CSV file for excel
print(df)
df.to_csv('simulation_2.csv', index=False)

Calculating n_particles = 50
Calculating n_particles = 60
Calculating n_particles = 70
Calculating n_particles = 80
Calculating n_particles = 90
Calculating n_particles = 100
Calculating n_particles = 110
Calculating n_particles = 120
Calculating n_particles = 130
Calculating n_particles = 140
Calculating n_particles = 150
Calculating n_particles = 160
Calculating n_particles = 170
Calculating n_particles = 180
Calculating n_particles = 190
Calculating n_particles = 200
    n_particles                                              D_msd  \
0            50  [0.060806917640415924, 0.0628399496381419, 0.0...   
1            60  [0.06014737860075277, 0.05993113873874751, 0.0...   
2            70  [0.06527487566800826, 0.05885931045498654, 0.0...   
3            80  [0.06151521520744109, 0.05620969606719006, 0.0...   
4            90  [0.0571805547957105, 0.055346296609908446, 0.0...   
5           100  [0.060815450104677295, 0.05918369801736328, 0....   
6           110  [0.062008141488805

In [ ]:
# @title Simulation 3. MSD/VCAF over time at different densities.

densities_list = np.arange(0.65,0.95,0.05)
n_steps = 20_000
#reps = 5
T = 0.5

# Initialize data storage
data = []

# Run the simulation for each temperature and repetition
for arho in densities_list:
    print(f"Running simulation for rho={arho}")

    # Run the simulation
    R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, arho, T, sample_freq)

    # Calculate MSD and VACF corrected lists
    msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)

    # Get diffusion coefficients
    D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)

    # Store results in data
    data.append({
        'Density': arho,
        'Time': np.linspace(0,1,len(msd_list_cor)),
        'D_msd_list': msd_list_cor,
        'D_vacf': vacf_list_cor
    })

# Convert to DataFrame
df = pd.DataFrame(data)

## Display the DataFrame or save to CSV file for excel
print(df)
#df.to_csv('simulation_3.csv', index=False)

In [ ]:
  # Plotting MSD vs Time for different densities
  plt.figure(figsize=(12, 6))
  font_size = 12
  temp = 0.5

  # Subplot for MSD
  plt.subplot(1, 2, 1)
  for i, row in df.iterrows():
      plt.plot(row['Time'], row['D_msd_list'], label=f"$\\rho$ = {row['Density']:.2f}")
  plt.xlabel('Time')
  plt.ylabel('MSD')
  plt.title(f'MSD vs Time for Different Densities, T = {temp}', fontsize=font_size)
  plt.legend()
  plt.grid(True)

  # Subplot for VACF
  plt.subplot(1, 2, 2)
  for i, row in df.iterrows():
      plt.plot(row['Time'], row['D_vacf'], label=f"$\\rho$ = {row['Density']:.2f}")
  plt.xlabel('Time')
  plt.ylabel('VACF')
  plt.title(f'VACF vs Time for Different Densities, T = {temp}', fontsize=font_size)
  plt.legend()
  plt.grid(True)

  plt.tight_layout()
  plt.show()

In [ ]:
print(D_msd)
print(D_vacf)

# Simulation 4

Jan: Look carefully at the plots in Simulation 4 when selecting values of rho  for which to compute D. Use only rho values where a linear fit accurately represents the MSD curve and where the area under the VACF curve in the plot represents the area of the entire curve from 0 to infinity

In [ ]:
# @title Simulation 4. MSD/VCAF over time at different temperatures.
def run(density):
  temperatures_list = np.arange(0.1,1,0.1)
  n_steps = 20_000
  #reps = 5

  # Initialize data storage
  data = []

  # Run the simulation for each temperature and repetition
  for T in temperatures_list:
      print(f"Running simulation for T={T}")

      # Run the simulation
      R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, density, T, sample_freq)

      # Calculate MSD and VACF corrected lists
      msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)

      # Get diffusion coefficients
      D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)

      # Store results in data
      data.append({
          'Temperature': T,
          'Time': np.linspace(0,1,len(msd_list_cor)),
          'D_msd_list': msd_list_cor,
          'D_vacf': vacf_list_cor
      })

  # Convert to DataFrame
  df = pd.DataFrame(data)

  ## Display the DataFrame or save to CSV file for excel
  #print(df)
  # df.to_csv('sim3_test1.csv', index=False)

  # Plotting MSD vs Time for different temperatures
  plt.figure(figsize=(8, 4))
  font_size = 9

  # Subplot for MSD
  plt.subplot(1, 2, 1)
  for i, row in df.iterrows():
      plt.plot(row['Time'], row['D_msd_list'], label=f"T = {row['Temperature']:.1f}")
  plt.xlabel('Time')
  plt.ylabel('MSD')
  plt.title(f'MSD vs Time for Different Temperatures, $\\rho$ = {density}', fontsize=font_size)
  plt.legend()
  plt.grid(True)

  # Subplot for VACF
  plt.subplot(1, 2, 2)
  for i, row in df.iterrows():
      plt.plot(row['Time'], row['D_vacf'], label=f"T = {row['Temperature']:.1f}")
  plt.xlabel('Time')
  plt.ylabel('VACF')
  plt.title(f'VACF vs Time for Different Temperatures, $\\rho$ = {density}', fontsize=font_size)
  plt.legend()
  plt.grid(True)

  plt.tight_layout()
  plt.show()

In [ ]:
run(0.8)

In [ ]:
run(0.8)

In [ ]:
# @title Kører run med forskellige densiteter
## Vi kører for nogle forskellige rho værdier og ser, at rho helst skal være omkring en 0.7-0.8 stykker.

run(0.1)
run(0.5)
run(0.7)
run(0.8)
run(1)
run(1.5)

## Vis output

Konklusion: Densiteten skal helst være omkring 0.7 eller 0.8. Hav dette i mente når man fortolker i simulation 5 og 6!

# Simulation 5 og 6

In [ ]:
# @title Simulation 5: Different densities
densities_list = np.arange(0.6,0.95,0.05)
n_steps = 20_000
reps = 10

# Initialize data storage
data = []

# Run the simulation for each temperature and repetition
for rho in densities_list:
    print(f"Running simulation for rho={rho}")

    D_msd_list = []
    D_vacf_list = []

    for rep in range(reps):

        # Run the simulation
        R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, rho, T, sample_freq)
        # Calculate MSD and VACF corrected lists
        msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)
        # Get diffusion coefficients
        D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)
        # Store results in data
        D_msd_list.append(D_msd_temp)
        D_vacf_list.append(D_vacf_temp)

    data.append({
            'Density': rho,
            'D_msd': D_msd_list,
            'D_vacf': D_vacf_list
        })

# Convert to DataFrame
df = pd.DataFrame(data)

## Display the DataFrame or save to CSV file for excel
print(df)
df.to_csv('simulation_5.csv', index=False)

Running simulation for rho=0.6
Running simulation for rho=0.65
Running simulation for rho=0.7000000000000001
Running simulation for rho=0.7500000000000001
Running simulation for rho=0.8000000000000002
Running simulation for rho=0.8500000000000002
Running simulation for rho=0.9000000000000002
   Density                                              D_msd  \
0     0.60  [0.0821035006695141, 0.07493076706700494, 0.07...   
1     0.65  [0.06932670632062826, 0.06997633781182265, 0.0...   
2     0.70  [0.06677337974116414, 0.06185793805230687, 0.0...   
3     0.75  [0.04732696292866084, 0.04665702762460072, 0.0...   
4     0.80  [0.03172053216141311, 0.0286708346839971, 0.03...   
5     0.85  [0.019204783332046695, 0.015387440914322878, 0...   
6     0.90  [0.00696485426038387, 0.008858737762644509, 0....   

                                              D_vacf  
0  [0.09231018259439562, 0.07841100368941967, 0.0...  
1  [0.06582211383742972, 0.07480672395818068, 0.0...  
2  [0.065491619089834

In [ ]:
# @title Simulation 6: Different temperatures
temperatures_list = np.arange(0.2, 4.2, 0.2)
n_steps = 20_000
reps = 10

# Initialize data storage
data = []

# Run the simulation for each temperature and repetition
for T in temperatures_list:
    print(f"Running simulation for T={T}")

    D_msd_list = []
    D_vacf_list = []

    for rep in range(reps):

        # Run the simulation
        R_list, V_list, t_list, n_samples = simulation(n_particles, n_steps, dt, rho, T, sample_freq)
        # Calculate MSD and VACF corrected lists
        msd_list_cor, vacf_list_cor = get_msd_vacf_cor(R_list, V_list)
        # Get diffusion coefficients
        D_msd_temp, D_vacf_temp = get_D(msd_list_cor, vacf_list_cor)

        D_msd_list.append(D_msd_temp)
        D_vacf_list.append(D_vacf_temp)

    # Store results in data
    data.append({
        'Temperature': T,
        'D_msd': D_msd_list,
        'D_vacf': D_vacf_list
    })

# Convert to DataFrame
df = pd.DataFrame(data)

## Display the DataFrame or save to CSV file for excel
print(df)
df.to_csv('simulation_6.csv', index=False)

Running simulation for T=0.2
Running simulation for T=0.4
Running simulation for T=0.6000000000000001
Running simulation for T=0.8
Running simulation for T=1.0
Running simulation for T=1.2
Running simulation for T=1.4000000000000001
Running simulation for T=1.6
Running simulation for T=1.8
Running simulation for T=2.0
Running simulation for T=2.2
Running simulation for T=2.4000000000000004
Running simulation for T=2.6000000000000005
Running simulation for T=2.8000000000000003
Running simulation for T=3.0000000000000004
Running simulation for T=3.2
Running simulation for T=3.4000000000000004
Running simulation for T=3.6000000000000005
Running simulation for T=3.8000000000000003
Running simulation for T=4.0
    Temperature                                              D_msd  \
0           0.2  [0.0025012197375543365, 0.0029774839707079758,...   
1           0.4  [0.00502340890726273, 0.007411465099015765, 0....   
2           0.6  [0.010447503708699615, 0.009467198630250254, 0...   
3    